# 트랙 3: PU Loss (na_weight=0.7) 풀 파이프라인 재현

> **최종 업데이트**: 2026-07-13: na_weight sweep에서 선정된 0.7로 distant 20,000개 프리트레인 →
> annotated 15 epoch 파인튜닝 풀 런. 비교군은 팀원 baseline `atlop`(동일 레시피·seed 66, PU만 꺼짐,
> dev F1 61.71 / Ign F1 59.86, `results/comparison.md`) — 새로 돌릴 필요 없음.

**실행 전**: 런타임 → 런타임 유형 변경 → **A100 GPU** 선택.

**판정 기준**: 마지막 epoch의 `dev_F1 / Ign_F1`이 **61.71 / 59.86보다 높으면** "PU 이득이
annotated 파인튜닝 후에도 유지된다"가 입증됨 (스테이지1 소규모 A/B에서는 +2.93 F1이었음 —
`Scripts/atlop/PU_THRESHOLD_EXPERIMENT.md` 참고).

In [ ]:
# 0) GPU 확인 (A100이 보여야 함)
!nvidia-smi -L

In [ ]:
# 1) 코드 + 데이터 (Git LFS에서 필요한 json만 선별로 당김 — LFS 대역폭 절약)
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/multiful/Information_Extraction.git
%cd Information_Extraction
!git lfs pull --include="docred_data/data/dev.json,docred_data/data/train_annotated.json,docred_data/data/train_distant.json,docred_data/data/rel_info.json"
# 아래에서 json들이 MB 단위(train_distant는 GB 근처)로 보이면 성공.
# 133바이트 같은 크기로 보이면 LFS pull 실패 -> Drive 백업본으로 우회 필요
!ls -lh docred_data/data/*.json

In [ ]:
# 2) 패키지 (Colab 기본 torch 사용, transformers 버전만 로컬 개발환경과 통일)
!pip install -q transformers==4.57.6 accelerate

In [ ]:
# 3) 학습: distant 20k PU(0.7) 프리트레인 -> annotated 15ep 파인튜닝 (A100 약 1~2시간)
#    팀원 baseline과 정확히 같은 레시피에 --use_pu_loss --na_weight 0.7만 추가.
#    epoch마다 [.. | epoch N] dev_F1=.. Ign_F1=.. 로그가 찍힘.
!python -m Scripts.atlop.train_re --epochs 15 --distant_limit 20000 --distant_epochs 1 \
  --use_pu_loss --na_weight 0.7 \
  --run_name atlop_full_pu07 --save_model --seed 66

In [ ]:
# 4) 결과 백업 (세션 끊기면 파일이 사라지므로 반드시 실행)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"
!cp results/atlop_full_pu07.pt results/atlop_full_pu07_stage1.pt \
   results/atlop_full_pu07_dev_predictions.json \
   "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"
!ls -lh "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"

## 결과 기록

학습이 끝나면 마지막 epoch 로그의 수치를 아래 표에 채워서
`results/comparison.md`와 `Scripts/atlop/PU_THRESHOLD_EXPERIMENT.md`에 반영:

| | dev F1 | Ign F1 | P | R |
|---|---|---|---|---|
| baseline (PU off, 팀원 `atlop`) | 61.71 | 59.86 | 66.08 | 57.89 |
| **PU na_weight=0.7 (이 노트북)** | | | | |

- stage-1(distant 직후) 로그 수치도 따로 기록해두면 "파인튜닝 전/후 이득 변화"를 보일 수 있음.
- seed 66 단일 시드 결과이므로, 차이가 ±1점 이내면 PRD §5의 시드 노이즈 주의사항 적용.